# # Assignment 4
### Why are we here?
Usually, when you use ChatGPT or Gemini, your request travels to a massive server farm, gets processed, and comes back. This requires:
1.  **Internet Connectivity** (Not available everywhere, e.g., remote farms, drones).
2.  **Privacy Compromise** (Your data leaves your device).
3.  **High Latency** (Round-trip time to the server).

**Edge AI** moves this intelligence *directly onto the device* (your Raspberry Pi).

### Today's Goal
**Modules you will run:**
1. **Thinking Cost** (latency, TTFT vs throughput, thermal throttling, Pareto tradeoffs)
2. **RAG** (PDF → chunking → embeddings → Chroma retrieval → transparent prompting)
3. **Multimodal AI** (vision model inference + optional multimodal-RAG)

> **Design principle:** *Nothing is a black box.* Every module prints the **prompt**, **retrieved chunks**, **similarity scores**, and **device telemetry** so you can see what happens **under the hood** on an edge device.


## Student Report Instructions

**Objective:** Summarize your findings from the Edge AI modules.

1. **Module 1 (Thinking Cost):**
   - Report the average **TTFT** and **Throughput** for the baseline model.
   - Describe how TTFT scaled as you increased the input context length. Did you notice any thermal throttling?
2. **Module 2 (RAG):**
   - Provide the answer generated by the RAG system for the question: "Where are the boys staying?".
   - Explain why the "Router filter" is useful in a RAG pipeline.
3. **Module 3 (Vision):**
   - Paste the description generated by the vision model for the provided image.
   - Compare the wall time of a vision inference versus a text-only inference.
"""


## NOTE: Please makes changes only where its mentioned     #TO DO IT YOURSELF

In [ ]:
!pip install Ollama
!pip install chromadb

```markdown
## Configuration

This is the **single configuration point** for the whole notebook. You can adjust the models, measurement samples, and other parameters here. Feel free to experiment with different `MODEL_TEXT` options like `phi3:3.8b` or `tinyllama`.
```

In [ ]:
# ---------------- CONFIGURATION ----------------
## TO DO IT YOURSELF, Play with different models here

MODEL_TEXT = "phi3:3.8b"                 # text LLM (e.g., phi3, tinyllama, llama3.2)
MODEL_VISION = "moondream"          # vision model (e.g., moondream)
EMBEDDING_MODEL = "nomic-embed-text"  # embedding model in Ollama

MEASUREMENT_SAMPLES = 5            # repeat count for statistics
WARMUP_RUNS = 1                    # warmup runs (excluded from stats)
COOLDOWN_S = 1.0                   # sleep between trials to reduce thermal carry-over

# If True, we default to shorter runs (recommended during class)
QUICK_MODE = True

print(f"Models: text={MODEL_TEXT}, vision={MODEL_VISION}, embed={EMBEDDING_MODEL}")

```markdown
## Setting up Ollama Server in Colab

Since Ollama needs a running server, I'll install and start it in the Colab environment, then pull the necessary models for our tasks. This ensures everything runs within Colab.
```

In [ ]:
# Install zstd, a dependency for Ollama installation
!apt-get install zstd -y

# Install Ollama server (x86_64 for Colab environment)
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in the background
import subprocess
import time

print("Starting Ollama server...")
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5) # Give the server a moment to start
print("Ollama server started.")

# Pull the models defined in configuration
print(f"Pulling text model: {MODEL_TEXT}")
!ollama pull {MODEL_TEXT}
print(f"Pulling embedding model: {EMBEDDING_MODEL}")
!ollama pull {EMBEDDING_MODEL}
print(f"Pulling vision model: {MODEL_VISION}")
!ollama pull {MODEL_VISION}

print("All required Ollama models pulled.")

<details>
<summary><b>Cell 0.0 — Quick start checklist</b> (click to expand)</summary>


- Confirms you are on **Raspberry Pi**, prints CPU/RAM/swap/disk.
- Probes for **vcgencmd** telemetry (temperature, throttling).
- Probes for **accelerators** (USB / PCIe enumeration).
- Checks that **Ollama** is reachable and prints available models.

**What to look for**
- Temperature rises under sustained load.


</details>

In [ ]:
# Core libs
import os, time, json, math, subprocess
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional
from IPython.display import display

import psutil
import numpy as np

# Optional libs (the notebook will still run if these are missing)
try:
    import pandas as pd
except Exception:
    pd = None

try:
    import matplotlib.pyplot as plt
except Exception as e:
    raise RuntimeError("matplotlib is required for this notebook.") from e

try:
    import seaborn as sns
except Exception:
    sns = None

try:
    import PyPDF2
except Exception:
    PyPDF2 = None

# Ollama + Chroma (assumed installed on the Pi image)
try:
    import ollama
except Exception as e:
    raise RuntimeError("ollama python package is required. If missing: pip install ollama") from e

try:
    import chromadb
except Exception as e:
    raise RuntimeError("chromadb is required. If missing: pip install chromadb") from e

# ---------------- CONFIGURATION ----------------
## TO DO IT YOURSELF, Play with different models here

# MODEL_TEXT = "phi3:3.8b"                 # text LLM (e.g., phi3, tinyllama, llama3.2)
# MODEL_VISION = "moondream"          # vision model (e.g., moondream)
# EMBEDDING_MODEL = "nomic-embed-text"  # embedding model in Ollama

# MEASUREMENT_SAMPLES = 5            # repeat count for statistics
# WARMUP_RUNS = 1                    # warmup runs (excluded from stats)
# COOLDOWN_S = 1.0                   # sleep between trials to reduce thermal carry-over

# # If True, we default to shorter runs (recommended during class)
# QUICK_MODE = True

print(" Imports loaded.")
print(f"Models: text={MODEL_TEXT}, vision={MODEL_VISION}, embed={EMBEDDING_MODEL}")

<details>
<summary><b>Cell 0.1 — Imports + configuration</b> (click to expand)</summary>


This is the **single configuration point** for the whole notebook.

You can change:
- `MODEL_TEXT` (LLM)
- `MODEL_VISION` (vision model)
- `EMBEDDING_MODEL` (text embedding model)
- `MEASUREMENT_SAMPLES` (benchmark repetitions)
- `COOLDOWN_S` (to reduce thermal carry-over)


</details>

In [ ]:
import os, sys, platform, subprocess, textwrap, json, time
from pathlib import Path

def _run(cmd):
    """Run a shell command safely and return stdout (or an error string)."""
    try:
        out = subprocess.check_output(cmd, stderr=subprocess.STDOUT).decode(errors="replace")
        return out.strip()
    except Exception as e:
        return f"[command failed] {cmd}: {e}"

print("Python:", sys.version.split()[0])
print("OS:", platform.platform())
print("Machine:", platform.machine())

# CPU model (Pi shows up in /proc/cpuinfo)
cpuinfo = _run(["bash","-lc","cat /proc/cpuinfo | egrep 'Model|Hardware|Revision' | head -n 5"])
print("\n--- CPU info ---")
print(cpuinfo if cpuinfo else "(unavailable)")

# Memory / Swap
mem = _run(["bash","-lc","free -h"])
print("\n--- Memory ---")
print(mem)

# Disk
disk = _run(["bash","-lc","df -h / | tail -n 1"])
print("\n--- Disk (root filesystem) ---")
print(disk)

# Basic accelerator probe
print("\n--- Accelerator probe ---")
print("lsusb (first 20 lines):")
print("\n".join(_run(["bash","-lc","lsusb | head -n 20"]).splitlines()))
print("\nPCIe devices (lspci, if present):")
print(_run(["bash","-lc","command -v lspci >/dev/null 2>&1 && lspci | head -n 30 || echo 'lspci not installed'"]))

# Raspberry Pi telemetry tools (optional)
print("\n--- Raspberry Pi telemetry ---")
print("vcgencmd:", _run(["bash","-lc","command -v vcgencmd || echo 'vcgencmd not found'"]))
if "not found" not in _run(["bash","-lc","command -v vcgencmd || echo 'not found'"]):
    print("temp:", _run(["bash","-lc","vcgencmd measure_temp"]))
    print("throttled:", _run(["bash","-lc","vcgencmd get_throttled"]))

# Ollama sanity check
print("\n--- Ollama ---")
print("ollama:", _run(["bash","-lc","command -v ollama || echo 'ollama not found'"]))
if "ollama not found" not in _run(["bash","-lc","command -v ollama || echo 'ollama not found'"]):
    print("ollama list:")
    print(_run(["bash","-lc","ollama list | head -n 30"]))


In [ ]:
def _safe_read(path: str) -> Optional[str]:
    try:
        return Path(path).read_text().strip()
    except Exception:
        return None

def _vcgencmd(cmd: str) -> Optional[str]:
    try:
        out = subprocess.check_output(["bash","-lc", f"vcgencmd {cmd}"], stderr=subprocess.STDOUT).decode().strip()
        return out
    except Exception:
        return None

def get_enhanced_system_stats() -> Dict[str, Any]:
    # Temperature + throttling (Pi-specific)
    temp_raw = _vcgencmd("measure_temp")
    throttled = _vcgencmd("get_throttled")
    volts = _vcgencmd("measure_volts core")

    # CPU frequency (Linux sysfs; may differ across distros)
    freq_khz = _safe_read("/sys/devices/system/cpu/cpu0/cpufreq/scaling_cur_freq")
    cpu_freq_mhz = (int(freq_khz) / 1000.0) if (freq_khz and freq_khz.isdigit()) else None

    # CPU utilization snapshot
    cpu_percent = psutil.cpu_percent(interval=0.1)

    # RAM (system + process)
    vm = psutil.virtual_memory()
    proc = psutil.Process()
    rss_mb = proc.memory_info().rss / (1024**2)

    stats = {
        "temp": temp_raw or "N/A",
        "throttled": throttled or "N/A",
        "core_volts": volts or "N/A",
        "cpu_freq_mhz": cpu_freq_mhz if cpu_freq_mhz is not None else "N/A",
        "cpu_percent": cpu_percent,
        "ram_used_mb": vm.used / (1024**2),
        "ram_percent": vm.percent,
        "proc_rss_mb": rss_mb,
        "swap_used_mb": psutil.swap_memory().used / (1024**2),
    }

    # Pretty print
    print(f" Temp: {stats['temp']} |  Throttle: {stats['throttled']} |  CPU Freq: {stats['cpu_freq_mhz']} MHz")
    print(f" RAM: {stats['ram_used_mb']:.0f} MB ({stats['ram_percent']}%) |  Proc RSS: {stats['proc_rss_mb']:.0f} MB |  Swap: {stats['swap_used_mb']:.0f} MB")
    return stats

_ = get_enhanced_system_stats()


<details>
<summary><b>Cell 0.3 — Ollama smoke test (text + embeddings + vision availability)</b> (click to expand)</summary>


- Runs a tiny generation on `MODEL_TEXT`
- Requests one embedding from `EMBEDDING_MODEL`
- Verifies `MODEL_VISION` is listed (vision model doesn’t run unless you test it)

check if everything is all right and we are good to ho.


</details>

In [ ]:
def ollama_smoke_test():
    print(" Text generation test...")
    r = ollama.generate(model=MODEL_TEXT, prompt="Answer with a single number: 2+2=")
    print("LLM:", r["response"].strip())

    print("\n Embedding test...")
    emb = ollama.embeddings(model=EMBEDDING_MODEL, prompt="hello world")["embedding"]
    print("Embedding dims:", len(emb), "| norm:", float(np.linalg.norm(np.array(emb))))

    print("\n Available models (first 20 lines):")
    try:
        out = subprocess.check_output(["bash","-lc","ollama list | head -n 20"]).decode()
        print(out.strip())
    except Exception as e:
        print("Could not run `ollama list`:", e)

ollama_smoke_test()


---
# Module 1 — The Thinking Cost (Under the Hood)
We measure **what the user feels** (TTFT) and **what the system pays** (throughput, RAM, throttling).


<details>
<summary><b>Cell 1.1 — (multi-run stats + cooldown)</b> (click to expand)</summary>


We’ll measure:
- **Wall time** and **CPU time**
- **RAM delta** (process RSS)
- **System telemetry** snapshot after each trial

Why multi-run?
- Edge devices vary run-to-run because of temperature + frequency scaling.


</details>

In [ ]:
from statistics import mean, stdev

def measure_trials(fn, *, trials: int = MEASUREMENT_SAMPLES, warmup: int = WARMUP_RUNS, cooldown_s: float = COOLDOWN_S):
    """Run fn() multiple times and return (results, metrics)."""
    # Warmup
    for _ in range(warmup):
        _ = fn()
        time.sleep(cooldown_s)

    # Timed runs
    wall_times, cpu_times, rss_deltas = [], [], []
    outputs = []
    proc = psutil.Process()

    for t in range(trials):
        rss0 = proc.memory_info().rss
        cpu0 = proc.cpu_times()
        t0 = time.perf_counter()

        out = fn()

        t1 = time.perf_counter()
        cpu1 = proc.cpu_times()
        rss1 = proc.memory_info().rss

        wall = t1 - t0
        cpu = (cpu1.user + cpu1.system) - (cpu0.user + cpu0.system)
        rss_delta = (rss1 - rss0) / (1024**2)

        wall_times.append(wall)
        cpu_times.append(cpu)
        rss_deltas.append(rss_delta)
        outputs.append(out)

        print(f"  trial {t+1}/{trials}: wall={wall:.3f}s | cpu={cpu:.3f}s | rssΔ={rss_delta:+.1f}MB")
        _ = get_enhanced_system_stats()
        time.sleep(cooldown_s)

    metrics = {
        "wall_mean": mean(wall_times),
        "wall_std": stdev(wall_times) if len(wall_times) > 1 else 0.0,
        "cpu_mean": mean(cpu_times),
        "cpu_std": stdev(cpu_times) if len(cpu_times) > 1 else 0.0,
        "rss_delta_mean_mb": mean(rss_deltas),
        "rss_delta_std_mb": stdev(rss_deltas) if len(rss_deltas) > 1 else 0.0,
        "trials": trials,
    }
    return outputs, metrics

def pretty_metrics(m: Dict[str, Any]):
    print("\nSummary")
    print(f"Wall: {m['wall_mean']:.3f}s ± {m['wall_std']:.3f}s")
    print(f"CPU : {m['cpu_mean']:.3f}s ± {m['cpu_std']:.3f}s")
    print(f"RSSΔ: {m['rss_delta_mean_mb']:+.1f}MB ± {m['rss_delta_std_mb']:.1f}MB")


<details>
<summary><b>Cell 1.2 — Baseline inference (single prompt) + telemetry</b> (click to expand)</summary>

- run multiple trials
- print stats + telemetry

Try changing the prompt length and observe:
- TTFT tends to increase
- memory deltas can vary


</details>

In [ ]:
PROMPT_BASELINE = "Explain the importance of Edge AI in exactly 2 sentences."

def run_baseline_once():
    resp = ollama.generate(model=MODEL_TEXT, prompt=PROMPT_BASELINE)
    return resp["response"]

outs, met = measure_trials(run_baseline_once, trials=3 if QUICK_MODE else MEASUREMENT_SAMPLES)
pretty_metrics(met)
print("\nSample output:\n", outs[-1][:400])


<details>
<summary><b>Cell 1.3 — Streaming telemetry: TTFT + TPOT + tokens/sec</b> (click to expand)</summary>


- **TTFT** (time to first token): perceived latency
- **TPOT** (time per output token): throughput/steady-state speed

Run it for prompts of different lengths and compare.


</details>

In [ ]:
def measure_streaming(prompt: str, *, model: str = MODEL_TEXT, max_tokens: int = 80) -> Dict[str, Any]:
    start = time.perf_counter()
    first_token_t = None
    token_timestamps = []
    out_text = []

    stream = ollama.generate(model=model, prompt=prompt, options={"num_predict": max_tokens}, stream=True)

    for chunk in stream:
        # chunk typically contains keys: 'response', 'done', ...
        txt = chunk.get("response", "")
        if txt:
            out_text.append(txt)

        now = time.perf_counter()
        if first_token_t is None:
            first_token_t = now
        token_timestamps.append(now)

    end = time.perf_counter()

    ttft = (first_token_t - start) if first_token_t else (end - start)
    total = end - start
    n_tokens_est = len("".join(out_text).split())  # token estimate (word-level proxy)
    tok_per_s = (n_tokens_est / total) if total > 0 else 0.0

    if len(token_timestamps) >= 2:
        tpot_vals = np.diff(np.array(token_timestamps))
        tpot_mean = float(np.mean(tpot_vals))
        tpot_std = float(np.std(tpot_vals))
    else:
        tpot_mean = tpot_std = 0.0

    metrics = {
        "prompt_words": len(prompt.split()),
        "ttft_s": float(ttft),
        "total_s": float(total),
        "output_words_est": n_tokens_est,
        "throughput_words_per_s": float(tok_per_s),
        "tpot_mean_s": tpot_mean,
        "tpot_std_s": tpot_std,
    }
    return metrics

prompts = [
    "Hi.",
    "Explain Edge AI in one sentence.",
    "Describe the challenges of running LLMs on Raspberry Pi including memory constraints and thermal throttling."
]

rows = []
for p in prompts:
    print("\n---")
    print("Prompt:", p)
    m = measure_streaming(p, max_tokens=80 if QUICK_MODE else 150)
    print(json.dumps(m, indent=2))
    rows.append(m)
    time.sleep(1.0)

if pd is not None:
    df_stream = pd.DataFrame(rows)
    display(df_stream)


<details>
<summary><b>Cell 1.4 — TTFT scaling with context length (why long prompts feel slow)</b> (click to expand)</summary>


We generate synthetic prompts of increasing length and measure **TTFT**.

Expected observation:
- TTFT grows faster-than-linear as context grows (attention prefill cost)
- On edge devices, throttling makes it *worse over time*



</details>

In [ ]:
def make_prompt(n_words: int) -> str:
    # A neutral, repeatable prompt generator (no external data needed)
    return ("Edge AI " * n_words).strip() + "\nExplain in one paragraph."

lengths = [20, 80, 200] if QUICK_MODE else [20, 80, 200, 400, 800]
records = []

for n in lengths:
    p = make_prompt(n)
    m = measure_streaming(p, max_tokens=40 if QUICK_MODE else 80)
    m["input_words"] = n
    records.append(m)
    print(f"n={n:4d} -> TTFT={m['ttft_s']:.3f}s | throughput={m['throughput_words_per_s']:.2f} words/s")
    time.sleep(1.0)

if pd is not None:
    df_ttft = pd.DataFrame(records)
else:
    df_ttft = records

# Plot
xs = [r["input_words"] for r in records]
ys = [r["ttft_s"] for r in records]

plt.figure(figsize=(7,4))
plt.scatter(xs, ys)
plt.xlabel("Input length (words, synthetic proxy)")
plt.ylabel("TTFT (seconds)")
plt.title("TTFT scaling on edge device")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


<details>
<summary><b>Cell 1.6 — Monitor Ollama server RAM (why your Python RSS is misleading)</b> (click to expand)</summary>

- LLM inference happens inside the **`ollama` server process**, not inside this notebook kernel.
- This cell finds the `ollama` PID(s) and lets us sample **RSS** (resident memory) over time.
- Later, we’ll use this to visualize **KV-cache / context** pressure under long prompts.

</details>

In [ ]:
import threading

def find_ollama_pids() -> List[int]:
    pids = []
    for p in psutil.process_iter(attrs=["pid","name","cmdline"]):
        try:
            name = (p.info.get("name") or "").lower()
            cmd = " ".join(p.info.get("cmdline") or []).lower()
            if "ollama" in name or cmd.startswith("ollama") or " ollama" in cmd:
                pids.append(p.info["pid"])
        except Exception:
            pass
    return sorted(set(pids))

def rss_mb(pid: int) -> float:
    try:
        return psutil.Process(pid).memory_info().rss / (1024**2)
    except Exception:
        return float("nan")

OLLAMA_PIDS = find_ollama_pids()
print("Ollama PIDs:", OLLAMA_PIDS if OLLAMA_PIDS else "(not found)")

if OLLAMA_PIDS:
    for pid in OLLAMA_PIDS:
        print(f"  pid={pid} rss={rss_mb(pid):.0f} MB")
else:
    print("If Ollama is installed but PID not found, try: `systemctl status ollama` or `ps aux | grep ollama`")


<details>
<summary><b>Cell 1.7 — KV-cache / context pressure experiment (measure Ollama peak RAM during long prompts)DO NOT RUN THIS CELL</b> (click to expand)</summary>

- We repeatedly run generation with increasing **input length** and sample **peak RSS** of the `ollama` process.
- On edge devices, long prompts grow the attention KV-cache inside the model and push RAM up.
- Watch how TTFT + peak RAM change together.

</details>

In [ ]:
def sample_peak_rss_during(fn, *, pid: int, interval_s: float = 0.05) -> Tuple[Any, float]:
    peak = 0.0
    stop = False
    result = None
    exc = None

    def sampler():
        nonlocal peak, stop
        while not stop:
            m = rss_mb(pid)
            if not math.isnan(m):
                peak = max(peak, m)
            time.sleep(interval_s)

    th = threading.Thread(target=sampler, daemon=True)
    th.start()
    try:
        result = fn()
    except Exception as e:
        exc = e
    finally:
        stop = True
        th.join(timeout=1.0)
    if exc:
        raise exc
    return result, peak

if not OLLAMA_PIDS:
    print("Ollama PID not found; skipping.")
else:
    pid = OLLAMA_PIDS[0]
    print("Using pid:", pid)

    lens = [50, 200, 600] if QUICK_MODE else [50, 200, 600, 1200]
    rows = []
    for n in lens:
        prompt = make_prompt(n)
        def _do():
            return ollama.generate(
                model=MODEL_TEXT,
                prompt=prompt,
                options={"num_predict": 60 if QUICK_MODE else 120},
                keep_alive="2m"  # try to keep the model resident to stabilize measurements
            )["response"]

        t0 = time.perf_counter()
        _, peak = sample_peak_rss_during(_do, pid=pid, interval_s=0.05)
        t1 = time.perf_counter()
        # TTFT is more precise with streaming; here we track total as a coarse signal
        total = t1 - t0

        rows.append({"input_words": n, "total_s": total, "ollama_peak_rss_mb": peak})
        print(f"n={n:4d} total={total:.2f}s peak_rss={peak:.0f}MB")
        time.sleep(1.0)

    if pd is not None:
        df_kv = pd.DataFrame(rows)
        display(df_kv)

    plt.figure(figsize=(7,4))
    plt.plot([r["input_words"] for r in rows], [r["ollama_peak_rss_mb"] for r in rows], marker="o")
    plt.xlabel("Input length (words)")
    plt.ylabel("Ollama peak RSS (MB)")
    plt.title("Peak Ollama RAM vs context length (proxy for KV-cache pressure)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


## Module 2: RAG (Retrieval Augmented Generation)
Standard LLMs (like Llama 3) rely on training data that is **cut off in the past**. They don't know about *this* specific Winter School because the brochure wasn't on the internet when they were trained.

To fix this, we use **RAG**. Think of RAG as an "Open Book Exam":
1.  **Ingestion (Study):** We break our private document (PDF) into small chunks.
2.  **Retrieval (Search):** When you ask a question, we find the specific page in the book that contains the answer.
3.  **Generation (Answer):** We show that page to the AI and say, *"Using this information, answer the question."*

![image.png](attachment:image.png)

<details>
<summary><b>Cell 2.0 — Load knowledge: PDF → text → overlapping chunks (fallback if no PDF)</b> (click to expand)</summary>


Real RAG systems ingest messy documents.

This cell:
- reads `winterschool.pdf` (if present)
- extracts text (PyPDF2)
- chunks it with overlap to preserve context
- falls back to the baseline facts if PDF is missing

Put `winterschool.pdf` in the same folder as this notebook to use real data.


</details>

In [ ]:
FALLBACK_DOCS = [

    "Event: ACM India Winter School on Edge AI.",
    "Dates: 28 December 2025 to 4 January 2026.",
    "Venue: CDS building, Indian Institute of Science (IISc), Bengaluru.",


    "About: The winter school provides an in-depth overview of software platforms, hardware systems, and AI models/algorithms for efficient deployment on accelerated and classic edge devices.",
    "Coverage: Edge computing architectures and accelerators; co-optimization of edge systems and ML models for performance, power and accuracy.",
    "Coverage: Federated learning frameworks; deployment of AI, Generative AI/LLM models and AI agents at the edge for practical IoT applications such as smart mobility and smart agriculture.",
    "Format: Lectures, hands-on sessions, and expert talks; participants gain practical skills to design, implement, and optimize intelligent edge systems for real-world applications.",


    "Key Topics: Foundations of IoT, Accelerated Edge Computing and Edge AI.",
    "Key Topics: Tiny and Embedded Machine Learning.",
    "Key Topics: Model Optimization and Acceleration for Edge AI.",
    "Key Topics: Edge AI Platforms, Frameworks, and Deployment Pipelines.",
    "Key Topics: Federated Learning and Distributed Training for Edge Devices.",
    "Key Topics: Neuromorphic Computing and Brain-Inspired Architectures.",
    "Key Topics: Generative AI and LLM at the Edge.",
    "Key Topics: Agentic AI on the Edge.",
    "Key Topics: Security, Privacy, and Responsible AI in Edge Systems.",
    "Key Topics: Benchmarking, Profiling, and Performance Evaluation of Edge AI Systems.",
    "Key Topics: Edge AI for Drones/UAVs, IoT, Smart Cities, and Industrial Applications.",


    "Keynote/Invited Speakers: Vijay Janappa Reddi (Harvard University, USA).",
    "Keynote/Invited Speakers: Sajal Das (Missouri University of Science and Technology, USA).",
    "Keynote/Invited Speakers: Varun Ojha (Newcastle University, UK).",
    "Keynote/Invited Speakers: Archan Misra (Singapore Management University).",
    "Keynote/Invited Speakers: Prashant Shenoy (University of Massachusetts).",
    "Technical Session Speakers include: Ajay Pratap (IIT (BHU), Varanasi); Gayathri Ananthanarayanan (IIT Dharwad); Manik Gupta (BITS Pilani); Pandarasamy Arjunan (IISc); Yogesh Simmhan (IISc).",
    "Technical Session Speakers include: Dr. TV Prabhakar (IISc); Punit Rathore (IISc); Chetan Singh Thakur (IISc); Sumit Kumar Mandal (IISc); Prashanti (AMD); Prasant Misra (TCS Research).",


    "School timing: 8:00 AM to 8:00 PM on all days.",
    "Meals: Breakfast 8:00–9:00 AM; tea/coffee/snacks morning and evening; lunch 12:45–2:00 PM; dinner 7:00–8:00 PM (provided daily to attendees).",


    "Day 1 (Sun, 28 Dec 2025) — Hardware Systems: Welcome Address by Pandarasamy Arjunan (IISc) and Yogesh Simmhan (IISc).",
    "Day 1 — Keynote: Vijay Janappa Reddi (Harvard) — 'Edge AI: Opportunities and Challenges'.",
    "Day 1 — Dr. TV Prabhakar (IISc) — 'In-Network Edge Intelligence for Tactile CPS' (morning sessions).",
    "Day 1 — Pandarasamy Arjunan (IISc) — 'Tiny ML' (afternoon sessions).",

    "Day 2 (Mon, 29 Dec 2025) — Hardware Systems: Adithya Krishna + Chetan Singh Thakur (IISc) — 'RAMAN: A Reconfigurable and Sparse tinyML Accelerator for Inference on Edge'.",
    "Day 2 — Gayathri Ananthanarayanan (IIT Dharwad) — 'Architecting Efficient Edge AI Systems: From Accelerators to Runtime Adaptation' (morning sessions).",
    "Day 2 — Pandarasamy Arjunan (IISc) — 'Embedded Computer Vision' (afternoon sessions).",

    "Day 3 (Tue, 30 Dec 2025) — Edge AI Platforms: Yogesh Simmhan (IISc) — 'ML Software Platforms' (morning sessions).",
    "Day 3 — Prashanti (AMD) — 'ML Software Platforms' (late morning session).",
    "Day 3 — Yogesh Simmhan (IISc) — 'ML on Edge Accelerators' (afternoon sessions).",
    "Day 3 — Prashant Shenoy (UMass) — 'Data Centers, AI Workloads, and Efficiency: An Edge Systems Perspective' (evening talk).",

    "Day 4 (Wed, 31 Dec 2025) — Federated Learning: Yogesh Simmhan (IISc) — 'Federated Learning' (morning sessions).",
    "Day 4 — Yogesh Simmhan (IISc) — 'FL on Edge' (late morning + evening sessions).",
    "Day 4 — Varun Ojha (Newcastle University) — 'Safeguarding Artificial Intelligence' (afternoon session).",
    "Day 4 — Varun Ojha (Newcastle University) — 'Resource-Efficient Artificial Intelligence' (late afternoon session).",

    "Day 5 (Thu, 1 Jan 2026) — Gen AI & Edge AI for Mobility: Sumit Kumar Mandal (IISc) — 'LLM Algorithms' (morning).",
    "Day 5 — Sumit Kumar Mandal (IISc) — 'LLMs at Edge' (morning).",
    "Day 5 — Prasant Misra (IISc & TCS) — 'Edge AI in Mobility' (late morning).",
    "Day 5 — Punit Rathore (IISc) — 'Edge AI in Mobility' (afternoon).",
    "Day 5 — Pandarasamy Arjunan (IISc) — 'Gen AI at Edge' (evening sessions).",

    "Day 6 (Fri, 2 Jan 2026) — Edge Analytics: Manik Gupta (BITS Pilani) — 'IoT Analytics' (sessions across the day).",
    "Day 6 — 'MATLAB for Edge AI' (late afternoon session).",
    "Day 6 — 'RBCCPS Tour' (tour session).",

    "Day 7 (Sat, 3 Jan 2026) — Edge AI in Agriculture: Ajay Pratap (IIT BHU) — 'UAV and Edge AI in Agriculture' (morning).",
    "Day 7 — Archan Misra (SMU) — 'Optimizing Edge Execution of AI-based Machine Perception' (late morning).",
    "Day 7 — 'Arm for Edge AI' (afternoon session).",
    "Day 7 — Hackathon (late afternoon + evening).",

    "Day 8 (Sun, 4 Jan 2026) — Hackathon: Sajal Das (Missouri S&T) — 'Smart Connected Farms: AI and IoT-based Pest Management in Precision Agriculture' (morning).",
    "Day 8 — Hackathon + Demo/Presentation sessions; Certificate Distribution and Closing Ceremony (evening).",


    "Venue & Access: IISc campus is accessible from Bengaluru; nearest metro station is Yeshwanthpur.",


    "Girls’ Accommodation: Hoysala Guest House (IISc). Address: Opposite NMR Research Centre (Next to CCE Building), Student Council Road, IISc Campus, Devasandra Layout, Bengaluru, Karnataka 560012.",
    "Girls’ Accommodation: Reception Number: 08022932535.",
    "Boys’ Accommodation: Sri Durga Hotel. Address: 35/1, Gayathri Temple Rd, Dr. Ambedkar Nagar, Yeshwanthapura, Bengaluru, Karnataka 560022.",
    "Boys’ Accommodation: Phone Number: 09019158456.",


    "Program Coordinators (Contact): Pandarasamy Arjunan (IISc) — samy@iisc.ac.in.",
    "Program Coordinators (Contact): Yogesh Simmhan (IISc) — simmhan@iisc.ac.in.",
    "Sponsor: ACM India Council."
]

def extract_and_chunk_pdf(pdf_path: str, chunk_chars: int = 700, overlap_chars: int = 120) -> Optional[List[str]]:
    if PyPDF2 is None:
        print("PyPDF2 not installed -> cannot parse PDF. Using fallback.")
        return None
    p = Path(pdf_path)
    if not p.exists():
        print(f"PDF not found at: {p.resolve()} -> Using fallback.")
        return None

    try:
        with p.open("rb") as f:
            reader = PyPDF2.PdfReader(f)
            pages = []
            for page in reader.pages:
                txt = page.extract_text() or ""
                pages.append(txt)
        full = "\n".join(pages).replace("\x00"," ")
        full = " ".join(full.split())

        chunks = []
        i = 0
        while i < len(full):
            chunk = full[i:i+chunk_chars]
            chunks.append(chunk)
            i += max(1, chunk_chars - overlap_chars)
        chunks = [c.strip() for c in chunks if c.strip()]
        print(f"Extracted {len(chunks)} chunks from PDF.")
        return chunks
    except Exception as e:
        print("PDF extraction failed:", e)
        return None

pdf_chunks = extract_and_chunk_pdf("winterschool.pdf", chunk_chars=800 if QUICK_MODE else 1200, overlap_chars=150)
winter_school_docs = pdf_chunks if pdf_chunks else FALLBACK_DOCS
print(f"Loaded {len(winter_school_docs)} knowledge chunks.")
print("Example chunk:\n", winter_school_docs[0][:250], "...")


<details>
<summary><b>Cell 2.1 — Build vector DB (Ollama embeddings → Chroma) + inspect vectors</b> (click to expand)</summary>


We explicitly compute embeddings using **Ollama** (`EMBEDDING_MODEL`) and store them in **Chroma**.

Under-the-hood:
- You’ll see embedding dimensionality + norms.
- This demystifies “semantic search” as just geometry in vector space.


</details>

![image.png](attachment:image.png)

In [ ]:
import re
import numpy as np
import chromadb
from typing import Any, Dict, List, Optional


# winter_school_docs = FALLBACK_DOCS # Removed this line to allow PDF parsing from previous cell


def l2_normalize(vec: List[float]) -> List[float]:
    v = np.array(vec, dtype=np.float32)
    n = float(np.linalg.norm(v) + 1e-12)
    return (v / n).tolist()


def bucketize(doc: str) -> str:
    s = doc.lower()
    if any(k in s for k in ["accommodation", "hotel", "guest house", "boys’ accommodation", "girls’ accommodation"]):
        return "logistics"
    if any(k in s for k in ["day 1", "day 2", "day 3", "day 4", "day 5", "day 6", "day 7", "day 8", "school timing", "meals"]):
        return "schedule"
    if any(k in s for k in ["key topics", "foundations", "tiny", "embedded", "federated", "neuromorphic", "generative", "agentic", "security", "benchmarking"]):
        return "topics"
    if any(k in s for k in ["speaker", "keynote", "technical session", "university", "iit", "iisc", "tcs", "amd"]):
        return "speakers"
    if any(k in s for k in ["contact", "@iisc.ac.in", "program coordinators", "sponsor"]):
        return "contacts"
    return "misc"

# Build Chroma collection in COSINE space + explicit embeddings

COLLECTION_NAME = "winter_school_curated_cosine"

client = chromadb.Client()

# clean slate
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

docs = winter_school_docs
ids = [f"doc_{i}" for i in range(len(docs))]
metas = [{"i": i, "bucket": bucketize(docs[i]), "source": "curated"} for i in range(len(docs))]

# Compute embeddings with SAME model used at query time
embs = []
for d in docs:
    e = ollama.embeddings(model=EMBEDDING_MODEL, prompt=d)["embedding"]
    embs.append(l2_normalize(e))  # normalize for cosine stability

collection.add(
    ids=ids,
    documents=docs,
    metadatas=metas,
    embeddings=embs
)

print(f"Built collection '{COLLECTION_NAME}' with {len(docs)} docs (cosine space, explicit embeddings).")

<details>
<summary><b>Cell 2.1b — Inspect embeddings (raw text → first dimensions → norms)</b> (click to expand)</summary>

- you’ll see how a chunk becomes a long vector.
- We print the first 8 dimensions for a few chunks and the vector norm.
- Different chunks have different directions (semantics), but similar norms.

</details>

In [ ]:
def inspect_embeddings(docs: List[str], embs: List[List[float]], k: int = 3, dims: int = 8):
    for i in range(min(k, len(docs))):
        v = np.array(embs[i], dtype=float)
        print(f"\nChunk {i} preview: {docs[i][:]}")
        print(f"  first {dims} dims:", np.array2string(v[:dims], precision=3, separator=", "))
        print(f"  norm:", float(np.linalg.norm(v)))

inspect_embeddings(winter_school_docs, embs, k=3, dims=8) # Changed doc_embs to embs

<details>
<summary><b>Cell 2.2 — retrieval + prompt construction + answer generation</b> (click to expand)</summary>


It prints:
- retrieved chunks
- similarity distances (lower is closer)
- exact prompt sent to the model

If the answer is not in the retrieved context, we instruct the model to say **NOT FOUND IN CONTEXT**.


</details>

In [ ]:
def route_where(question: str) -> Optional[Dict[str, Any]]:
    q = question.lower()
    if any(k in q for k in ["boys", "girls", "stay", "staying", "accommodation", "hotel", "guest house"]):
        return {"bucket": "logistics"}
    if any(k in q for k in ["day", "schedule", "timing", "when", "breakfast", "lunch", "dinner"]):
        return {"bucket": "schedule"}
    if any(k in q for k in ["topic", "covers", "syllabus", "what will", "learn"]):
        return {"bucket": "topics"}
    if any(k in q for k in ["speaker", "keynote", "talk", "who is"]):
        return {"bucket": "speakers"}
    if any(k in q for k in ["email", "contact", "coordinator"]):
        return {"bucket": "contacts"}
    return None

def retrieve(question: str, top_k: int = 5) -> Dict[str, Any]:
    q_emb = ollama.embeddings(model=EMBEDDING_MODEL, prompt=question)["embedding"]
    q_emb = l2_normalize(q_emb)

    where = route_where(question)

    res = collection.query(
        query_embeddings=[q_emb],
        n_results=top_k,
        where=where,
        include=["documents", "distances", "metadatas"]
    )

    docs = res["documents"][0]
    dists = res["distances"][0]   # cosine distance: ~0 (best) to ~2 (worst)
    metas = res["metadatas"][0]
    return {"docs": docs, "dists": dists, "metas": metas, "where": where}


def rag_answer(question: str, top_k: int = 3, *, model: str = MODEL_TEXT) -> Dict[str, Any]:
    r = retrieve(question, top_k=top_k)

    print(f"\nQuestion: {question}")
    print(f"Router filter(where) = {r['where']}")
    print(f"Retrieved top-{top_k} chunks:")
    for i, (doc, dist, meta) in enumerate(zip(r["docs"], r["dists"], r["metas"]), start=1):
        print(f"  {i}. cosine_dist={dist:.4f} bucket={meta.get('bucket')} i={meta.get('i')} :: {doc[:100]}...")

    context = "\n\n".join([f"[{i}] {d}" for i, d in enumerate(r['docs'], start=1)])


    prompt = f"""You are a precise assistant for the Winter School.

RULES (must follow):
1) Use ONLY the numbered context chunks below.
2) If the answer is not explicitly present, reply exactly: NOT FOUND IN CONTEXT
3) Output ONLY the final answer in ONE line. No extra text, no examples, no follow-up questions.


Context:
{context}

Question: {question}

Final Answer (one line only):"""


    resp = ollama.generate(
        model=model,
        prompt=prompt,
        options={
            "num_predict": 60,
            "temperature": 0.1,
            "top_p": 0.9,
            "stop": ["\n\n", "Question:", "Context:", "RULES", "Final Answer:"]
        }
    )

    answer = resp["response"].strip()


    answer = answer.splitlines()[0].strip()

    return {"answer": answer, "prompt": prompt, **r}


out = rag_answer("Where are the boys staying?", top_k=5)
print("\nAnswer:", out["answer"].strip())


<details>
<summary><b>Cell 2.3 — Visualize retrieval geometry (PCA projection)</b> (click to expand)</summary>


We project high-dimensional embeddings to 2D using PCA to show:
- the query vector
- the top-k retrieved document vectors

This makes “nearest neighbors” feel real.

(Requires scikit-learn.)


</details>

In [ ]:
def visualize_vector_space(question: str, top_k: int = 5):
    try:
        from sklearn.decomposition import PCA
    except Exception as e:
        print("scikit-learn not installed. Install with: pip install scikit-learn")
        return

    r = retrieve(question, top_k=top_k)
    q_emb = np.array(ollama.embeddings(model=EMBEDDING_MODEL, prompt=question)["embedding"])
    doc_embs = np.array([ollama.embeddings(model=EMBEDDING_MODEL, prompt=d)["embedding"] for d in r["docs"]])

    X = np.vstack([q_emb, doc_embs])
    pca = PCA(n_components=2)
    X2 = pca.fit_transform(X)

    plt.figure(figsize=(7,6))
    plt.scatter(X2[1:,0], X2[1:,1], s=80, label="docs")
    for i in range(top_k):
        plt.annotate(f"doc{i+1}", (X2[i+1,0], X2[i+1,1]))
    plt.scatter(X2[0,0], X2[0,1], s=160, marker="*", label="query")
    plt.annotate("query", (X2[0,0], X2[0,1]))
    plt.title("RAG retrieval geometry (PCA projection)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

visualize_vector_space("When is the hackathon?", top_k=3)


## Module 3: Multimodal AI (Vision)
Humans don't just read; we see. **Multimodal models** can process text *and* images simultaneously.

We are using **Moondream**, a tiny computer-vision model designed specifically for edge devices like laptops and Pis.

**How it works:**
1.  The image is broken into "patches" (small squares).
2.  These patches are converted into tokens (numbers), just like text.
3.  The model "reads" the image patches and generates a text description.

<details>
<summary><b>Cell 3.0 — Image utilities (inspect, display, downscale)</b> (click to expand)</summary>


Before running a vision model, inspect the input:
- file size
- resolution
- format

On edge devices, huge images waste CPU and RAM.


</details>

In [ ]:
from PIL import Image

def inspect_image(path: str) -> Dict[str, Any]:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Image not found: {p.resolve()}")
    size_kb = p.stat().st_size / 1024
    img = Image.open(p)
    info = {
        "path": str(p),
        "size_kb": size_kb,
        "mode": img.mode,
        "format": img.format,
        "width": img.width,
        "height": img.height
    }
    print(json.dumps(info, indent=2))
    return info

def downscale_image(src: str, dst: str, max_side: int = 768):
    img = Image.open(src)
    w,h = img.size
    scale = min(max_side / max(w,h), 1.0)
    new_size = (int(w*scale), int(h*scale))
    img2 = img.resize(new_size)
    img2.save(dst)
    print(f"Saved downscaled image to {dst} ({new_size[0]}x{new_size[1]})")

IMAGE_PATH = "/content/place5.JPG"

if Path(IMAGE_PATH).exists():
    _ = inspect_image(IMAGE_PATH)
    display(Image.open(IMAGE_PATH))
else:
    print(f"{IMAGE_PATH} not found. Put an image next to this notebook or update IMAGE_PATH.")


<details>
<summary><b>Cell 3.1 — Vision inference (Moondream) with performance + telemetry</b> (click to expand)</summary>


This runs the vision model via `ollama.chat()` with image bytes.

Under the hood, we measure:
- wall time
- CPU time
- RSS delta
and print telemetry after each trial.


</details>

In [ ]:
def analyze_image_once(image_path: str, prompt: str) -> str:
    p = Path(image_path)
    if not p.exists():
        return f"Image not found: {p.resolve()}"
    with p.open("rb") as f:
        img_bytes = f.read()

    resp = ollama.chat(
        model=MODEL_VISION,
        messages=[{
            "role": "user",
            "content": prompt,
            "images": [img_bytes]
        }]
    )
    return resp["message"]["content"]

VISION_PROMPT = "Describe this image in 2-3 sentences. Then list 3 visible objects."

if Path(IMAGE_PATH).exists():
    def _fn():
        return analyze_image_once(IMAGE_PATH, VISION_PROMPT)

    outs, met = measure_trials(_fn, trials=2 if QUICK_MODE else 3)
    pretty_metrics(met)
    print("\n Vision output (last run):\n", outs[-1])
else:
    print("Provide IMAGE_PATH first.")



## Student Report Instructions

**Objective:** Summarize your findings from the Edge AI modules.

1. **Module 1 (Thinking Cost):**
   - Report the average **TTFT** and **Throughput** for the baseline model.
   - Describe how TTFT scaled as you increased the input context length. Did you notice any thermal throttling?
2. **Module 2 (RAG):**
   - Provide the answer generated by the RAG system for the question: "Where are the boys staying?".
   - Explain why the "Router filter" is useful in a RAG pipeline.
3. **Module 3 (Vision):**
   - Paste the description generated by the vision model for the provided image.
   - Compare the wall time of a vision inference versus a text-only inference.
"""
